In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

bronze_adls_path = "abfss://bronze@"+storage_account+".dfs.core.windows.net/policy_products/"
silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/policy_products/"

In [0]:
from pyspark.sql.types import *

policy_schema = StructType([

    StructField("policy_product_id", LongType(), True),
    StructField("policy_code", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("product_category", StringType(), True),

    StructField("plan_type", StringType(), True),
    StructField("coverage_type", StringType(), True),
    StructField("policy_term_years", IntegerType(), True),

    StructField("minimum_age", IntegerType(), True),
    StructField("maximum_age", IntegerType(), True),

    StructField("base_premium_amount", DoubleType(), True),
    StructField("premium_frequency", StringType(), True),

    StructField("coverage_amount", DoubleType(), True),
    StructField("deductible_amount", DoubleType(), True),
    StructField("co_payment_percentage", IntegerType(), True),
    StructField("no_claim_bonus_percentage", IntegerType(), True),

    StructField("max_claim_limit_per_year", DoubleType(), True),
    StructField("lifetime_claim_limit", DoubleType(), True),

    StructField("smoker_loading_percentage", IntegerType(), True),
    StructField("bmi_loading_threshold", IntegerType(), True),
    StructField("bmi_loading_percentage", IntegerType(), True),

    StructField("pre_existing_loading_percentage", IntegerType(), True),
    StructField("pre_existing_waiting_period_months", IntegerType(), True),

    StructField("maternity_cover_flag", BooleanType(), True),
    StructField("critical_illness_cover_flag", BooleanType(), True),

    StructField("product_launch_date", DateType(), True),
    StructField("product_status", StringType(), True),

    StructField("underwriting_required_flag", BooleanType(), True),

    StructField("created_at", TimestampType(), True)
])

In [0]:
policy_df = spark.read \
.option("header","true") \
.schema(policy_schema) \
.csv(bronze_adls_path)

In [0]:
from pyspark.sql.functions import *

policy_df_silver = (
    policy_df

    # clean text
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("product_category", trim(col("product_category")))
    .withColumn("plan_type", trim(col("plan_type")))
    .withColumn("coverage_type", trim(col("coverage_type")))
    .withColumn("product_status", trim(col("product_status")))

    # product age
    .withColumn(
        "product_age_years",
        floor(datediff(current_date(), col("product_launch_date")) / 365)
    )

    # premium band
    .withColumn(
        "premium_band",
        when(col("base_premium_amount") < 20000, "Low")
        .when(col("base_premium_amount") < 80000, "Medium")
        .otherwise("High")
    )

    # coverage band
    .withColumn(
        "coverage_band",
        when(col("coverage_amount") <= 500000, "Basic")
        .when(col("coverage_amount") <= 2000000, "Standard")
        .otherwise("High Coverage")
    )

    # risk factor indicator
    .withColumn(
        "high_risk_product_flag",
        when(
            (col("smoker_loading_percentage") >= 20) |
            (col("pre_existing_loading_percentage") >= 40),
            1
        ).otherwise(0)
    )
)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy(col("policy_product_id")).orderBy(col("created_at").desc())

policy_df_silver = (
    policy_df_silver
    .withColumn("rnk", rank().over(window))
    .filter(col("rnk")==1)
    .drop(col("rnk"))
)

In [0]:
policy_df_silver_final = policy_df_silver.select(

    "policy_product_id",
    "policy_code",
    "product_name",
    "product_category",

    "plan_type",
    "coverage_type",
    "policy_term_years",

    "minimum_age",
    "maximum_age",

    "base_premium_amount",
    "premium_band",
    "premium_frequency",

    "coverage_amount",
    "coverage_band",

    "deductible_amount",
    "co_payment_percentage",
    "no_claim_bonus_percentage",

    "max_claim_limit_per_year",
    "lifetime_claim_limit",

    "smoker_loading_percentage",
    "bmi_loading_threshold",
    "bmi_loading_percentage",

    "pre_existing_loading_percentage",
    "pre_existing_waiting_period_months",

    "maternity_cover_flag",
    "critical_illness_cover_flag",

    "high_risk_product_flag",

    "product_launch_date",
    "product_age_years",
    "product_status",

    "underwriting_required_flag",

    "created_at"
)

policy_df_silver_final.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save(silver_adls_path)

In [0]:
spark.sql(f"""
OPTIMIZE delta.`{silver_adls_path}`
ZORDER BY (policy_product_id)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,